In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

DATA_DIR = '../data'

unified_alerts = pd.read_csv(f'{DATA_DIR}/unified_alerts.csv', parse_dates=[
    'window_start', 'inflow_timestamp', 'hop1_timestamp'
])
risk_profiles = pd.read_csv(f'{DATA_DIR}/customer_risk_profiles.csv')
peer_anomaly = pd.read_csv(f'{DATA_DIR}/peer_anomaly_scores.csv')
ground_truth = pd.read_csv(f'{DATA_DIR}/ground_truth.csv')
accounts = pd.read_csv(f'{DATA_DIR}/accounts.csv')

print(f"Unified alerts: {unified_alerts.shape}")
print(f"Risk profiles: {risk_profiles.shape}")
print(f"Peer anomaly scores: {peer_anomaly.shape}")
print(f"Ground truth: {ground_truth.shape}")

unified_alerts.head()

Unified alerts: (366, 14)
Risk profiles: (10000, 29)
Peer anomaly scores: (120000, 15)
Ground truth: (1576, 6)


,customer_id,alert_type,severity,status,suppression_reason,window_start,total_amount,num_deposits,inflow_timestamp,inflow_amount,outflow_ratio,hop1_timestamp,hop1_amount,hop3_to_country
0,CUST_102615,Structuring (1-day),High,Active,NaN,2023-03-11 14:00:00,92854.57,2.0,NaT,NaN,NaN,NaT,NaN,NaN
1,CUST_102615,Structuring (1-day),High,Active,NaN,2023-03-12 13:00:00,87206.60,2.0,NaT,NaN,NaN,NaT,NaN,NaN
2,CUST_103058,Structuring (1-day),High,Active,NaN,2023-05-29 15:00:00,72442.71,2.0,NaT,NaN,NaN,NaT,NaN,NaN
3,CUST_103265,Structuring (1-day),High,Active,NaN,2023-03-06 12:00:00,62108.55,2.0,NaT,NaN,NaN,NaT,NaN,NaN
4,CUST_103265,Structuring (1-day),High,Active,NaN,2023-03-08 14:00:00,68471.48,2.0,NaT,NaN,NaN,NaT,NaN,NaN


In [2]:
unified_alerts['alert_date'] = unified_alerts['window_start'].fillna(
    unified_alerts['inflow_timestamp']
).fillna(
    unified_alerts['hop1_timestamp']
)

print(unified_alerts['alert_date'].isna().sum(), "alerts with no resolvable date")
unified_alerts[['customer_id', 'alert_type', 'alert_date']].head()

0 alerts with no resolvable date


,customer_id,alert_type,alert_date
0,CUST_102615,Structuring (1-day),2023-03-11 14:00:00
1,CUST_102615,Structuring (1-day),2023-03-12 13:00:00
2,CUST_103058,Structuring (1-day),2023-05-29 15:00:00
3,CUST_103265,Structuring (1-day),2023-03-06 12:00:00
4,CUST_103265,Structuring (1-day),2023-03-08 14:00:00


In [3]:
#extracting month and year to match with peer_anomaly
unified_alerts['alert_year_month'] = unified_alerts['alert_date'].dt.to_period('M').astype(str)
peer_anomaly['year_month'] = peer_anomaly['year_month'].astype(str)

unified_alerts[['customer_id', 'alert_type', 'alert_date', 'alert_year_month']].head()

,customer_id,alert_type,alert_date,alert_year_month
0,CUST_102615,Structuring (1-day),2023-03-11 14:00:00,2023-03
1,CUST_102615,Structuring (1-day),2023-03-12 13:00:00,2023-03
2,CUST_103058,Structuring (1-day),2023-05-29 15:00:00,2023-05
3,CUST_103265,Structuring (1-day),2023-03-06 12:00:00,2023-03
4,CUST_103265,Structuring (1-day),2023-03-08 14:00:00,2023-03


In [4]:
### We now want to aggregate these alerts for each customers
severity_rank = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
unified_alerts['severity_rank'] = unified_alerts['severity'].map(severity_rank)

customer_alerts = unified_alerts.groupby('customer_id').agg(
    num_triggers=('alert_type', 'count'),
    alert_types=('alert_type', lambda x: list(x.unique())),
    max_severity_rank=('severity_rank', 'max'),
    alert_months=('alert_year_month', lambda x: list(x.unique()))
).reset_index()

customer_alerts['max_severity'] = customer_alerts['max_severity_rank'].map(
    {v: k for k, v in severity_rank.items()}
)

print(f"Unique customers with at least one alert: {len(customer_alerts)}")
customer_alerts.head()

Unique customers with at least one alert: 220


,customer_id,num_triggers,alert_types,max_severity_rank,alert_months,max_severity
0,CUST_100239,1,[Structuring (7-day)],2,[2023-06],Medium
1,CUST_100251,1,[Rapid Movement],3,[2023-03],High
2,CUST_100302,2,[Structuring (7-day)],2,[2023-08],Medium
3,CUST_100311,1,[Rapid Movement],3,[2023-08],High
4,CUST_100330,1,[Structuring (7-day)],2,[2023-07],Medium


### Now I'm going to check if anyof these customers was anomaly flagged in the same month as any of their alerts.

In [5]:
anomaly_lookup = peer_anomaly.set_index(['customer_id', 'year_month'])['is_anomaly'].to_dict()

def check_anomaly_overlap(row):
    for ym in row['alert_months']:
        if anomaly_lookup.get((row['customer_id'], ym), False):
            return True
    return False

customer_alerts['anomaly_same_month'] = customer_alerts.apply(check_anomaly_overlap, axis=1)

print(customer_alerts['anomaly_same_month'].value_counts())

anomaly_same_month
False    203
True      17
Name: count, dtype: int64


Now I'm going to attach phase 2's consolidated risk rating. 

In [6]:
customer_alerts = customer_alerts.merge(
    risk_profiles[['customer_id', 'crr_tier', 'composite_crr']],
    on='customer_id', how='left'
)

customer_alerts[['customer_id', 'num_triggers', 'max_severity', 'anomaly_same_month', 'crr_tier', 'composite_crr']].head()

,customer_id,num_triggers,max_severity,anomaly_same_month,crr_tier,composite_crr
0,CUST_100239,1,Medium,False,High,0.456057
1,CUST_100251,1,High,False,High,0.423696
2,CUST_100302,2,Medium,False,High,0.450036
3,CUST_100311,1,High,False,High,0.477664
4,CUST_100330,1,Medium,False,High,0.514067


***I'm now going to build the weighted composit priority score. 
I'm going use the following scores and weights, max_severity_rank (1-4), composite_crr (0-1, roughly, from Phase 2), anomaly_same_month (boolean, 0/1)***
***Alert severity gets 50% weight, CRR 35% and anomaly detection 15%**

In [8]:
#Cause the severity score is on a different scale, I'm going to normalise it. 

In [9]:
customer_alerts['severity_norm'] = customer_alerts['max_severity_rank'] / 4

customer_alerts['priority_score'] = (
    0.5 * customer_alerts['severity_norm'] +
    0.35 * customer_alerts['composite_crr'] +
    0.15 * customer_alerts['anomaly_same_month'].astype(int)
)

customer_alerts['priority_score'].describe()

count    220.000000
mean       0.498542
std        0.099773
min        0.338392
25%        0.416596
50%        0.472343
75%        0.570454
max        0.780228
Name: priority_score, dtype: float64

I'm going to add one more feature to the priority score if all three alerts are fired for a customer. Their score would increase by 3%. This is because, the chances of 3 alets triggering all at the same time do indicate higher money laundering risks. 

In [10]:
customer_alerts['trigger_boost'] = np.clip((customer_alerts['num_triggers'] - 1) * 0.03, 0, 0.10)

customer_alerts['priority_score_final'] = customer_alerts['priority_score'] + customer_alerts['trigger_boost']

customer_alerts['priority_score_final'].describe()

count    220.000000
mean       0.514633
std        0.104038
min        0.351874
25%        0.437842
50%        0.498357
75%        0.574362
max        0.849052
Name: priority_score_final, dtype: float64

I'm going tier these risks into how urgent an analyst needs to review these alerts

In [11]:
for pct in [50, 70, 85, 95]:
    val = np.percentile(customer_alerts['priority_score_final'], pct)
    print(f"{pct}th percentile: {val:.3f}")

50th percentile: 0.498
70th percentile: 0.563
85th percentile: 0.645
95th percentile: 0.720


In [12]:
p85 = customer_alerts['priority_score_final'].quantile(0.85)
p50 = customer_alerts['priority_score_final'].quantile(0.50)

def assign_tier(score):
    if score >= p85:
        return 'P1 - Critical'
    elif score >= p50:
        return 'P2 - High'
    else:
        return 'P3 - Standard'

customer_alerts['triage_tier'] = customer_alerts['priority_score_final'].apply(assign_tier)

customer_alerts['triage_tier'].value_counts()

triage_tier
P3 - Standard    110
P2 - High         77
P1 - Critical     33
Name: count, dtype: int64

So most of the alerts triggered do not require immediate actions from analyst like we would expect. I'm now going to validate this against ground truth.

In [15]:
gt_type_counts = ground_truth.merge(accounts[['account_id','customer_id']], on='account_id') \
    .groupby('customer_id')['scenario_type'].nunique().rename('num_distinct_scenario_types')

customer_alerts_check = customer_alerts.merge(gt_type_counts, on='customer_id', how='left')
customer_alerts_check.groupby('triage_tier')['num_distinct_scenario_types'].mean()

triage_tier
P1 - Critical    1.125000
P2 - High        1.039474
P3 - Standard    1.027273
Name: num_distinct_scenario_types, dtype: float64

In [18]:
for tier in ['P1 - Critical', 'P2 - High', 'P3 - Standard']:
    subset = customer_alerts[customer_alerts['triage_tier'] == tier]
    all_types = [t for sublist in subset['alert_types'] for t in sublist]
    print(f"\n{tier} alert type composition:")
    print(pd.Series(all_types).value_counts(normalize=True).round(2))


P1 - Critical alert type composition:
Structuring (7-day)    0.35
Structuring (1-day)    0.29
Rapid Movement         0.24
Layering               0.12
Name: proportion, dtype: float64

P2 - High alert type composition:
Rapid Movement         0.63
Layering               0.19
Structuring (7-day)    0.17
Structuring (1-day)    0.01
Name: proportion, dtype: float64

P3 - Standard alert type composition:
Structuring (7-day)    1.0
Name: proportion, dtype: float64


In [17]:
customer_alerts['num_distinct_types'] = customer_alerts['alert_types'].apply(lambda x: len(set(x)))

customer_alerts['trigger_boost_v2'] = np.clip((customer_alerts['num_distinct_types'] - 1) * 0.05, 0, 0.10)

customer_alerts['priority_score_final_v2'] = customer_alerts['priority_score'] + customer_alerts['trigger_boost_v2']

p85_v2 = customer_alerts['priority_score_final_v2'].quantile(0.85)
p50_v2 = customer_alerts['priority_score_final_v2'].quantile(0.50)

customer_alerts['triage_tier_v2'] = customer_alerts['priority_score_final_v2'].apply(
    lambda s: 'P1 - Critical' if s >= p85_v2 else ('P2 - High' if s >= p50_v2 else 'P3 - Standard')
)

for tier in ['P1 - Critical', 'P2 - High', 'P3 - Standard']:
    subset = customer_alerts[customer_alerts['triage_tier_v2'] == tier]
    all_types = [t for sublist in subset['alert_types'] for t in sublist]
    print(f"\n{tier} alert type composition:")
    print(pd.Series(all_types).value_counts(normalize=True).round(2))


P1 - Critical alert type composition:
Structuring (7-day)    0.28
Rapid Movement         0.26
Structuring (1-day)    0.24
Layering               0.22
Name: proportion, dtype: float64

P2 - High alert type composition:
Rapid Movement         0.60
Structuring (7-day)    0.21
Layering               0.14
Structuring (1-day)    0.05
Name: proportion, dtype: float64

P3 - Standard alert type composition:
Structuring (7-day)    1.0
Name: proportion, dtype: float64


### Refining the Trigger Boost: A Design Flaw Found and Corrected

The initial validation of the triage ranking showed an unexpected result. P1 Critical alerts were dominated by Structuring, which made up 64 percent of the combined alerts, while the more severe Layering typology made up only 12 percent.

After checking the scoring formula, the reason became clear. The trigger count boost was based on the total number of alerts per customer. This created a problem because Structuring uses a rolling window and can produce multiple overlapping alerts from one underlying pattern, as identified in Phase 3. In comparison, Layering and Rapid Movement usually trigger only once for each genuine event. This meant the scoring was unintentionally rewarding repeated alerts from the same rule instead of customers supported by multiple different detection signals.

This was corrected by changing the boost to use the number of distinct alert types per customer instead of the total alert count. This better represents genuine confirmation across different and independent detection rules. After the change, Layering's share in P1 increased from 12 percent to 22 percent, showing a meaningful improvement.

P1 is now more evenly distributed across all four alert types, with each making up around 22 to 28 percent. This is considered the expected result rather than another problem with the ranking.

The priority score is designed as a combination of several factors, including typology severity, the independent Customer Risk Rating or CRR, and the presence of other behavioral anomalies. It is not meant to rank customers based on alert severity alone. Because of this, a customer with a lower severity alert but strong supporting risk signals can correctly rank above a customer with a more severe alert but weaker overall risk context. This is the intended result of a scoring approach that combines multiple sources of risk rather than simply sorting alerts by severity.


In [19]:
####Some final clean ups

In [20]:
customer_alerts['priority_score'] = customer_alerts['priority_score_final_v2']
customer_alerts['triage_tier'] = customer_alerts['triage_tier_v2']

final_cols = [
    'customer_id', 'num_triggers', 'num_distinct_types', 'alert_types', 'max_severity',
    'crr_tier', 'composite_crr', 'anomaly_same_month', 'priority_score', 'triage_tier'
]

triage_output = customer_alerts[final_cols].sort_values('priority_score', ascending=False)
triage_output.head(10)

,customer_id,num_triggers,num_distinct_types,alert_types,max_severity,crr_tier,composite_crr,anomaly_same_month,priority_score,triage_tier
74,CUST_103265,7,2,"[Structuring (1-day), Structuring (7-day)]",High,High,0.640150,True,0.799052,P1 - Critical
5,CUST_100372,1,1,[Layering],Critical,Medium,0.372080,True,0.780228,P1 - Critical
129,CUST_105594,1,1,[Layering],Critical,Medium,0.353782,True,0.773824,P1 - Critical
10,CUST_100653,1,1,[Layering],Critical,Medium,0.272326,True,0.745314,P1 - Critical
79,CUST_103523,1,1,[Layering],Critical,Medium,0.270009,True,0.744503,P1 - Critical
84,CUST_103840,1,1,[Rapid Movement],High,High,0.626621,True,0.744317,P1 - Critical
177,CUST_107773,1,1,[Rapid Movement],High,High,0.594721,True,0.733152,P1 - Critical
161,CUST_107039,1,1,[Rapid Movement],High,High,0.590506,True,0.731677,P1 - Critical
103,CUST_104602,1,1,[Rapid Movement],High,High,0.556581,True,0.719803,P1 - Critical
201,CUST_108814,1,1,[Rapid Movement],High,High,0.530851,True,0.710798,P1 - Critical


### Phase 5 Summary: Alert Scoring, Prioritization and Triage

Combined the outputs from Phase 2, customer risk ratings, Phase 3, typology detection alerts, and Phase 4, peer anomaly detection, into a single customer level triage table. The aim was to address alert fatigue, where a large number of alerts and limited analyst capacity require a clear and defensible way to decide which customers should be investigated first.

**Methodology:**

* 220 unique customers with at least one Phase 3 alert were combined into one row per customer. The table captures trigger count, distinct alert types, highest alert severity, CRR tier, and same month anomaly co occurrence from Phase 4.

* A weighted priority score combined alert severity at 50 percent, customer CRR at 35 percent, and anomaly co occurrence at 15 percent. Alert severity received the highest weight because a direct match to a specific detection rule is the strongest evidence, while CRR and anomaly signals provide additional risk context.

* A trigger diversity boost was added for customers flagged by multiple distinct detection rules. This corrected an initial design flaw where using the total alert count unfairly favored Structuring, since its rolling window logic can naturally generate multiple alerts from the same underlying pattern. Using distinct alert types instead better rewards customers with supporting signals from different detection rules.

* Customers were assigned to P1 Critical, P2 High, and P3 Standard tiers using the 85th and 50th percentile cutoffs of the final priority score. This follows the same data driven threshold approach used throughout the project.

**Validation and limitations:**

The ground truth True and False label was not useful for directly validating the ranking. This is because all customers in the triage table had already triggered a Phase 3 detection rule, and most were already true positive cases regardless of their priority tier.

A second check looked at the number of distinct ground truth scenario types linked to each customer. This showed only a weak difference between higher and lower priority tiers. The main reason was a limitation of the synthetic dataset, where scenarios were mostly generated as independent and self contained events. Very few customers were involved in multiple scenario types, leaving limited variation for this test.

The more meaningful validation was therefore checking whether the scoring logic behaved as intended. P3 mainly isolates customers with single, lower severity Structuring alerts, while P1 contains a more balanced mix of alert types. This is expected because the final score does not depend on severity alone. It combines severity with customer risk rating and behavioral anomaly context, allowing customers with multiple supporting risk signals to rank higher even when their individual alert type is not the most severe.

The final output from this phase is the prioritized customer alert queue used directly in Phase 6 to power the Streamlit investigation dashboard.


In [21]:
triage_output.to_csv('../data/triage_queue.csv', index=False)
triage_output.shape

(220, 10)

In [23]:
# Thanks, I might not do phase 6. 